# EcoConnectAI — UNB7 final training on a cloud GPU (Colab / Kaggle)

**MODE B — FULL EXPERIMENT.** This notebook trains the paper's primary model — **UNB7 = U-Net decoder + EfficientNet-B7 encoder** — on the multi-area Sentinel-1 (VV/VH) tile dataset with Global Mangrove Watch weak labels, evaluates it on held-out spatial-block test tiles, calibrates the probability threshold, and packages the outputs for the repository.

Result label produced: **OUR EXPERIMENTAL RESULT** (agreement with the GMW weak label, not field-truth accuracy). Development (B0) results must never be relabelled as UNB7.

Requirements: a CUDA GPU with ≥ 16 GB (Colab T4/L4/A100 or Kaggle P100/T4). The dataset (`ecoconnect_tiles/`, built locally by `scripts/run_all_areas.py --stage tiles`) is uploaded to Google Drive or attached as a Kaggle dataset — the notebook never downloads imagery itself.

In [ ]:
# 1. Repository + dependencies (~2 min)
import os, subprocess, sys
REPO = "https://github.com/kuldeep31016/Major-Project-EconnectAI.git"   # <- your repo URL
if not os.path.exists("Major-Project-EconnectAI"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("Major-Project-EconnectAI")
!pip -q install "segmentation-models-pytorch>=0.5" timm rasterio pyproj shapely scipy pyyaml matplotlib scikit-image tqdm
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. Dataset location. EITHER mount Drive (Colab) OR point at the Kaggle input folder.
#    Expected: ${DATA_ROOT}/ecoconnect_tiles/{tiles/images,tiles/masks,splits,metadata.json}
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/ecoconnect_data"       # <- adjust
except ImportError:
    DATA_ROOT = "/kaggle/input/ecoconnect-tiles"                # <- adjust (Kaggle dataset name)
os.environ["DATA_ROOT"] = DATA_ROOT
!python scripts/inspect_dataset.py --path "$DATA_ROOT/ecoconnect_tiles" --max-files 3

In [ ]:
# 3. Train UNB7 (E1: Sentinel-1 VV/VH only — the paper's reported model).  configs/train_full.yaml
#    Edit epochs / batch size there or override with ECO_TRAINING__EPOCHS etc.  Checkpoints resume with --resume.
EXP = "all4_E1_s1_unb7_full"
!python scripts/train.py --config configs/train_full.yaml --experiment-id $EXP

In [ ]:
# 4. Evaluate on the held-out test split (dataset-level IoU/Dice/P/R/OA/kappa + qualitative panels)
!python scripts/evaluate.py --checkpoint outputs/segmentation/$EXP/best_model.pth --n-panels 12 --rgb-bands 4,3,2

In [ ]:
# 5. (optional, needs the study-area scenes + labels under $DATA_ROOT) whole-scene inference + pooled threshold sweep
#    Scenes are ~150 MB each; copy data/scenes and data/labels to Drive if you want this step here,
#    otherwise run the sweep locally with the downloaded checkpoint (scripts/run_all_areas.py --stage sweep).
import glob
scenes = sorted(glob.glob(f"{DATA_ROOT}/scenes/*/*_2020_s12_10m.tif"))
print(len(scenes), "scenes found")
if scenes:
    !python scripts/run_all_areas.py --stage sweep --experiment-id $EXP

In [ ]:
# 6. Package outputs for the repository (best_model.pth is ~270 MB for B7 — keep it out of git)
!zip -r -q /content/$EXP.zip outputs/segmentation/$EXP -x "*.pth" && ls -la /content/$EXP.zip
!cp outputs/segmentation/$EXP/best_model.pth "$DATA_ROOT/${EXP}_best_model.pth" 2>/dev/null || true
print("Copy the zip into outputs/segmentation/ locally, then run:\n  python scripts/run_all_areas.py --stage analyse --experiment-id", EXP, "--result-kind experiment")

## Ablations (same notebook, other configs)

| Exp | Config | Input |
|---|---|---|
| E2 | `configs/train_full_s2.yaml` | Sentinel-2 only |
| E3 | `configs/train_full_s1s2.yaml` | S1 + S2 early fusion (experimental; not the paper's model) |

Report every run with its `metrics.json` / `experiment.json`; the registry `outputs/segmentation/experiments.csv` gets one row per run.